# Pratica — Lezione 4. Programmazione dinamica

Questa pratica è tutta tua. Lavorerai in autonomia sullo stesso `GridWorld` usato in classe e implementerai da zero i due pilastri della **programmazione dinamica**: l'iterazione delle politiche e l'iterazione dei valori.

**Come lavorare con questo notebook:**
1. Esegui prima la sezione "Preparazione", che ricostruisce l'ambiente e le funzioni della lezione precedente
2. In ogni esercizio, completa le righe contrassegnate con `# TODO`
3. Esegui la tua soluzione e confronta il risultato con quanto descritto nella consegna
4. Solo dopo, apri la soluzione nascosta per verificare la tua risposta

> **Non guardare la soluzione prima di averci provato.** Il valore di questo esercizio sta nel confrontarsi un po' con il miglioramento delle politiche e con l'equazione di ottimalità di Bellman, non nel leggere la risposta.

## Preparazione: recupera l'ambiente della lezione

Queste celle riproducono esattamente il `GridWorld`, `politica_aleatoria`, `evaluacion_iterativa_politica` e `imprimir_politica` che conosci già dalle due lezioni precedenti. L'unica novità è un parametro nuovo e opzionale in `GridWorld`: `obstaculos`, una lista di caselle che l'agente non può attraversare —se tenta di entrarci, resta nella sua posizione, esattamente come se urtasse il bordo della griglia—. Di default vale `None` e il GridWorld si comporta esattamente come in classe.

Gli esercizi di oggi riutilizzano questi stessi nomi, quindi non cambiarli.

In [ ]:
import numpy as np

ACCIONES = ["arriba", "abajo", "izquierda", "derecha"]

DESPLAZAMIENTOS = {
    "arriba": (-1, 0),
    "abajo": (1, 0),
    "izquierda": (0, -1),
    "derecha": (0, 1),
}

FLECHAS = {"arriba": "^", "abajo": "v", "izquierda": "<", "derecha": ">"}

In [ ]:
class GridWorld:
    """
    Un mundo-cuadrícula de n_filas x n_columnas con una casilla terminal (la meta)
    y, opcionalmente, casillas-obstáculo.

    - Los estados son tuplas (fila, columna); los obstáculos no son estados
      habitables, así que no aparecen en mundo.estados.
    - Las transiciones son deterministas: moverse contra un muro, un obstáculo
      o el borde del tablero deja al agente en la misma casilla.
    - La recompensa es 'recompensa_paso' en cada movimiento (por defecto -1),
      salvo en la casilla terminal, donde el episodio ya ha terminado.
    """

    def __init__(self, n_filas=4, n_columnas=4, estados_terminales=None,
                 recompensa_paso=-1.0, obstaculos=None):
        self.n_filas = n_filas
        self.n_columnas = n_columnas
        self.recompensa_paso = recompensa_paso

        if estados_terminales is None:
            estados_terminales = [(n_filas - 1, n_columnas - 1)]
        self.estados_terminales = set(estados_terminales)
        self.obstaculos = set(obstaculos) if obstaculos else set()

        # Los obstáculos no son casillas habitables: el agente nunca "está" en ellas
        self.estados = [
            (f, c) for f in range(n_filas) for c in range(n_columnas)
            if (f, c) not in self.obstaculos
        ]
        self.acciones = ACCIONES

    def es_terminal(self, estado):
        return estado in self.estados_terminales

    def estado_siguiente(self, estado, accion):
        """Devuelve la casilla resultante de aplicar 'accion' desde 'estado'.
        Si el movimiento choca con el borde del tablero o con un obstáculo,
        el agente se queda quieto."""
        if self.es_terminal(estado):
            return estado

        df, dc = DESPLAZAMIENTOS[accion]
        fila, columna = estado
        nueva_fila = max(0, min(self.n_filas - 1, fila + df))
        nueva_columna = max(0, min(self.n_columnas - 1, columna + dc))
        nuevo_estado = (nueva_fila, nueva_columna)

        if nuevo_estado in self.obstaculos:
            return estado  # el obstáculo actúa como un muro
        return nuevo_estado

    def recompensa(self, estado, accion, estado_siguiente):
        if self.es_terminal(estado):
            return 0.0
        return self.recompensa_paso


def politica_aleatoria(mundo):
    """Construye una política estocástica que, en cada estado, elige
    cualquiera de las cuatro acciones con la misma probabilidad (0.25)."""
    probabilidad_uniforme = 1.0 / len(mundo.acciones)
    return {
        estado: {accion: probabilidad_uniforme for accion in mundo.acciones}
        for estado in mundo.estados
    }


def evaluacion_iterativa_politica(mundo, politica, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """Aplica la ecuación de Bellman de expectación barrido tras barrido,
    hasta que Vpi deja de cambiar. Devuelve V y el número de barridos usados."""
    V = {estado: 0.0 for estado in mundo.estados}
    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)
        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue
            valor_esperado = 0.0
            for accion in mundo.acciones:
                prob_accion = politica[estado][accion]
                estado_sig = mundo.estado_siguiente(estado, accion)
                r = mundo.recompensa(estado, accion, estado_sig)
                valor_esperado += prob_accion * (r + gamma * V[estado_sig])
            V_nuevo[estado] = valor_esperado
            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))
        V = V_nuevo
        if delta_maximo < umbral:
            break
    return V, iteracion


def imprimir_politica(mundo, politica):
    """Muestra la política como una cuadrícula de flechas. Las casillas
    obstáculo se marcan con '#' y la meta con 'G'."""
    for f in range(mundo.n_filas):
        fila_textos = []
        for c in range(mundo.n_columnas):
            estado = (f, c)
            if estado in mundo.obstaculos:
                fila_textos.append(" # ")
            elif mundo.es_terminal(estado):
                fila_textos.append(" G ")
            else:
                mejor_accion = max(politica[estado], key=politica[estado].get)
                fila_textos.append(f" {FLECHAS[mejor_accion]} ")
        print("|".join(fila_textos))

### Linea di base: il GridWorld originale

Esegui la cella seguente per avere a portata di mano lo stesso riferimento visto in classe: il GridWorld 4×4 senza ostacoli, con l'obiettivo nell'angolo in basso a destra. Userai `mundo` e `gamma` nell'esercizio 1, quindi non sovrascriverli.

In [ ]:
mundo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0)
gamma = 0.9

politica_inicial = politica_aleatoria(mundo)
V_inicial, barridos_iniciales = evaluacion_iterativa_politica(mundo, politica_inicial, gamma=gamma)

print(f"Vπ de la política aleatoria convergió en {barridos_iniciales} barridos.")
imprimir_politica(mundo, politica_inicial)

---

## Esercizio 1: completa l'iterazione delle politiche da zero

In classe hai costruito `mejora_politica` e `iteracion_politicas` vedendole già completate. Ora tocca a te completarle, sul `mundo` della linea di base.

**Suggerimenti:**
- In `mejora_politica_ejercicio`, per ogni azione ti servono lo stato successivo (`mundo.estado_siguiente`), la ricompensa immediata (`mundo.recompensa`) e il valore di un passo da salvare in `valores_q[accion]`: `r + gamma * V[estado_sig]`. Una volta costruito `valores_q`, `max(valores_q, key=valores_q.get)` ti restituisce l'azione con il valore più alto
- In `iteracion_politicas_ejercicio`, il ciclo alterna due chiamate —valutare con `evaluacion_iterativa_politica` e migliorare con `mejora_politica_ejercicio`— e termina quando la nuova politica è esattamente uguale a quella precedente (`nueva_politica == politica`)
- Con γ = 0,9 dovresti ottenere lo stesso risultato visto in classe: tre cicli e 90 passate totali, con la politica che punta sempre verso l'obiettivo lungo il percorso più breve

In [ ]:
def mejora_politica_ejercicio(mundo, V, gamma=0.9):
    """Construye una política determinista, codiciosa respecto a V: en cada
    estado, elige la acción que maximiza recompensa + gamma * V(siguiente)."""
    nueva_politica = {}
    for estado in mundo.estados:
        if mundo.es_terminal(estado):
            nueva_politica[estado] = {a: 0.0 for a in mundo.acciones}
            continue

        valores_q = {}
        for accion in mundo.acciones:
            # TODO: calcula el estado siguiente con mundo.estado_siguiente(estado, accion)
            estado_sig = None

            # TODO: calcula la recompensa inmediata con mundo.recompensa(estado, accion, estado_sig)
            r = None

            # TODO: guarda en valores_q[accion] el valor de un paso: r + gamma * V[estado_sig]
            valores_q[accion] = 0

        # TODO: escoge mejor_accion, la clave de valores_q con el valor más alto
        #       (pista: max(diccionario, key=diccionario.get))
        mejor_accion = None

        nueva_politica[estado] = {a: (1.0 if a == mejor_accion else 0.0) for a in mundo.acciones}
    return nueva_politica


def iteracion_politicas_ejercicio(mundo, gamma=0.9, umbral=1e-4):
    """Alterna evaluación completa y mejora hasta que la política deja de cambiar."""
    politica = politica_aleatoria(mundo)
    ciclos = 0
    barridos_totales = 0

    while True:
        ciclos += 1

        # TODO: evalúa la política actual con evaluacion_iterativa_politica(mundo, politica, gamma, umbral)
        V, barridos = None, None
        barridos_totales += barridos

        # TODO: mejora la política con mejora_politica_ejercicio(mundo, V, gamma)
        nueva_politica = None

        # TODO: si nueva_politica es igual a politica, la política ya no cambia: sal del bucle con break
        pass

        politica = nueva_politica

    return politica, V, ciclos, barridos_totales


politica_ej1, V_ej1, ciclos_ej1, barridos_ej1 = iteracion_politicas_ejercicio(mundo, gamma=gamma)

print(f"Ciclos: {ciclos_ej1}, barridos totales: {barridos_ej1}")
imprimir_politica(mundo, politica_ej1)

<details>
<summary>💡 Vedi la soluzione</summary>

```python
def mejora_politica_ejercicio(mundo, V, gamma=0.9):
    """Costruisce una politica deterministica, greedy rispetto a V: in ogni
    stato sceglie l'azione che massimizza ricompensa + gamma * V(successivo)."""
    nueva_politica = {}
    for estado in mundo.estados:
        if mundo.es_terminal(estado):
            nueva_politica[estado] = {a: 0.0 for a in mundo.acciones}
            continue

        valores_q = {}
        for accion in mundo.acciones:
            # Stato raggiunto dopo aver applicato l'azione
            estado_sig = mundo.estado_siguiente(estado, accion)

            # Ricompensa immediata di quella transizione
            r = mundo.recompensa(estado, accion, estado_sig)

            # Valore di un passo: ricompensa immediata + valore scontato del stato successivo
            valores_q[accion] = r + gamma * V[estado_sig]

        # L'azione greedy: quella che massimizza il valore di un passo
        mejor_accion = max(valores_q, key=valores_q.get)

        nueva_politica[estado] = {a: (1.0 if a == mejor_accion else 0.0) for a in mundo.acciones}
    return nueva_politica


def iteracion_politicas_ejercicio(mundo, gamma=0.9, umbral=1e-4):
    """Alterna valutazione completa e miglioramento finché la politica smette di cambiare."""
    politica = politica_aleatoria(mundo)
    ciclos = 0
    barridos_totales = 0

    while True:
        ciclos += 1

        # Valutazione completa della politica attuale
        V, barridos = evaluacion_iterativa_politica(mundo, politica, gamma, umbral)
        barridos_totales += barridos

        # Miglioramento greedy rispetto alla V appena calcolata
        nueva_politica = mejora_politica_ejercicio(mundo, V, gamma)

        # Se la politica non cambia più, vale l'equazione di ottimalità di Bellman
        if nueva_politica == politica:
            break

        politica = nueva_politica

    return politica, V, ciclos, barridos_totales


politica_ej1, V_ej1, ciclos_ej1, barridos_ej1 = iteracion_politicas_ejercicio(mundo, gamma=gamma)

print(f"Cicli: {ciclos_ej1}, passate totali: {barridos_ej1}")
imprimir_politica(mundo, politica_ej1)

# Output atteso con gamma = 0.9:
# Cicli: 3, passate totali: 90
#  v | v | v | v
#  v | v | v | v
#  v | v | v | v
#  > | > | > | G
```

**Perché funziona:** ogni ciclo dentro `while True` fa esattamente quello che hai fatto a mano in classe, cella per cella: valutare fino in fondo e poi migliorare. La condizione `nueva_politica == politica` è il criterio di arresto dell'iterazione delle politiche: quando l'azione greedy in ogni stato coincide già con quella che la politica eseguiva, vale l'equazione di ottimalità di Bellman e la politica è ottimale.
</details>

---

## Esercizio 2: completa l'iterazione dei valori e confronta su un GridWorld con ostacolo

Ora implementi la seconda componente di oggi: l'iterazione dei valori. La differenza chiave rispetto a `evaluacion_iterativa_politica` è che, invece di fare una media secondo una politica, ogni stato prende il **massimo** tra le sue azioni —l'equazione di ottimalità di Bellman—.

Prima costruisci il GridWorld con un ostacolo, poi completa il ciclo dell'iterazione dei valori. Alla fine confronta il numero di iterazioni necessarie per convergere con quelle di `iteracion_politicas_ejercicio` dell'esercizio precedente, eseguita sullo stesso GridWorld con ostacolo.

**Suggerimenti:**
- L'ostacolo si passa come `obstaculos=[(1, 1), (2, 1)]`: un muro verticale di due caselle al centro della griglia. La politica ottimale dovrà aggirarlo
- Dentro il ciclo, per ogni stato calcoli `valores_q`, una lista con il valore di un passo per ciascuna azione (`r + gamma * V[estado_sig]`), e aggiorni `V_nuevo[estado]` con il massimo di quella lista: `max(valores_q)`
- Al termine dell'iterazione su V, ricava la politica finale con un'unica chiamata a `mejora_politica_ejercicio(mundo, V, gamma)` —un ultimo sguardo di un passo, non serve più iterare—
- Confronta `ciclos_obs` (dell'iterazione delle politiche) e `barridos_obs_pi` (passate totali) con `barridos_obs_vi` (passate dell'iterazione dei valori): dovresti vedere lo stesso schema della classe, molte passate economiche contro pochi cicli costosi

In [ ]:
# Un obstáculo: un muro vertical de dos casillas en medio del tablero
obstaculos = [(1, 1), (2, 1)]
mundo_obstaculo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0, obstaculos=obstaculos)

print(f"Estados habitables: {len(mundo_obstaculo.estados)} de {4 * 4} casillas totales")


def iteracion_valores_ejercicio(mundo, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """Funde evaluación y mejora: cada barrido actualiza V con el máximo
    sobre acciones (ecuación de optimalidad de Bellman), no con un promedio
    según una política fija."""
    V = {estado: 0.0 for estado in mundo.estados}

    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)

        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue

            valores_q = []
            for accion in mundo.acciones:
                estado_sig = mundo.estado_siguiente(estado, accion)
                r = mundo.recompensa(estado, accion, estado_sig)
                # TODO: añade a valores_q el valor de un paso de esta acción: r + gamma * V[estado_sig]
                pass

            # TODO: actualiza V_nuevo[estado] con el máximo de valores_q (ecuación de optimalidad de Bellman)
            V_nuevo[estado] = 0.0

            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))

        V = V_nuevo
        if delta_maximo < umbral:
            break

    # TODO: deriva la política final con una última mirada de un paso:
    #       mejora_politica_ejercicio(mundo, V, gamma)
    politica_final = None

    return politica_final, V, iteracion


# Iteración de políticas sobre el GridWorld con obstáculo (reutilizas el ejercicio 1)
politica_obs_pi, V_obs_pi, ciclos_obs, barridos_obs_pi = iteracion_politicas_ejercicio(mundo_obstaculo, gamma=gamma)
print(f"\nIteración de políticas -> ciclos: {ciclos_obs}, barridos totales: {barridos_obs_pi}")
imprimir_politica(mundo_obstaculo, politica_obs_pi)

# Iteración de valores sobre el mismo GridWorld con obstáculo
politica_obs_vi, V_obs_vi, barridos_obs_vi = iteracion_valores_ejercicio(mundo_obstaculo, gamma=gamma)
print(f"\nIteración de valores -> barridos: {barridos_obs_vi}")
imprimir_politica(mundo_obstaculo, politica_obs_vi)

diferencia_maxima = max(abs(V_obs_pi[s] - V_obs_vi[s]) for s in mundo_obstaculo.estados)
print(f"\nDiferencia máxima entre las dos V*: {diferencia_maxima:.6f}")

<details>
<summary>💡 Vedi la soluzione</summary>

```python
obstaculos = [(1, 1), (2, 1)]
mundo_obstaculo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0, obstaculos=obstaculos)

print(f"Stati abitabili: {len(mundo_obstaculo.estados)} su {4 * 4} caselle totali")


def iteracion_valores_ejercicio(mundo, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """Unisce valutazione e miglioramento: ogni passata aggiorna V con il massimo
    sulle azioni (equazione di ottimalità di Bellman), non con una media
    secondo una politica fissa."""
    V = {estado: 0.0 for estado in mundo.estados}

    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)

        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue

            valores_q = []
            for accion in mundo.acciones:
                estado_sig = mundo.estado_siguiente(estado, accion)
                r = mundo.recompensa(estado, accion, estado_sig)
                # Valore di un passo di questa azione
                valores_q.append(r + gamma * V[estado_sig])

            # Equazione di ottimalità di Bellman: il massimo tra le azioni, non una media
            V_nuevo[estado] = max(valores_q)

            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))

        V = V_nuevo
        if delta_maximo < umbral:
            break

    # Un ultimo sguardo di un passo per ricavare la politica ottimale
    politica_final = mejora_politica_ejercicio(mundo, V, gamma)

    return politica_final, V, iteracion


politica_obs_pi, V_obs_pi, ciclos_obs, barridos_obs_pi = iteracion_politicas_ejercicio(mundo_obstaculo, gamma=gamma)
print(f"\nIterazione delle politiche -> cicli: {ciclos_obs}, passate totali: {barridos_obs_pi}")
imprimir_politica(mundo_obstaculo, politica_obs_pi)

politica_obs_vi, V_obs_vi, barridos_obs_vi = iteracion_valores_ejercicio(mundo_obstaculo, gamma=gamma)
print(f"\nIterazione dei valori -> passate: {barridos_obs_vi}")
imprimir_politica(mundo_obstaculo, politica_obs_vi)

diferencia_maxima = max(abs(V_obs_pi[s] - V_obs_vi[s]) for s in mundo_obstaculo.estados)
print(f"\nDifferenza massima tra le due V*: {diferencia_maxima:.6f}")

# Output atteso con gamma = 0.9:
# Stati abitabili: 14 su 16 caselle totali
#
# Iterazione delle politiche -> cicli: 3, passate totali: 91
#  v | > | v | v
#  v | # | v | v
#  v | # | v | v
#  > | > | > | G
#
# Iterazione dei valori -> passate: 7
#  v | > | v | v
#  v | # | v | v
#  v | # | v | v
#  > | > | > | G
#
# Differenza massima tra le due V*: 0.000000
```

**Perché funziona:** l'ostacolo riduce gli stati abitabili da 16 a 14 e obbliga la politica ad aggirarlo —nota la casella (0, 1), che ora punta a destra invece che in basso, proprio per schivare il muro—. Nonostante questo cambiamento nella mappa, lo schema dei costi resta identico a quello della classe: l'iterazione delle politiche richiede pochi cicli (tre), ma ciascuno è costoso (91 passate totali), mentre l'iterazione dei valori arriva allo stesso risultato in pochissime passate (sette). Entrambi i metodi approdano alla stessa politica ottimale, perché risolvono esattamente la stessa equazione di ottimalità di Bellman.
</details>

---

## Per continuare a esercitarti

Se ti è avanzato del tempo, torna ai due esercizi e prova queste varianti del capitolo:

1. **Cambia il fattore di sconto.** Prova con γ = 0,5 e con γ = 0,99 su `mundo_obstaculo`. Cambia il numero di passate che impiega ciascun metodo per convergere?
2. **Aggiungi un secondo ostacolo o un secondo obiettivo.** Combina `obstaculos` con una lista di due `estados_terminales` diversi e osserva come si ripartisce la griglia tra i due obiettivi
3. **Ingrandisci la scacchiera.** Prova con `n_filas=6, n_columnas=6` e un ostacolo più lungo. La differenza di passate tra i due metodi cresce o resta uguale?

**Prossima tappa:** nel prossimo blocco del corso lasci alle spalle l'ipotesi di conoscere il modello completo dell'ambiente (P e R), e impari a valutare e migliorare le politiche solo a partire dall'esperienza, con Monte Carlo e differenze temporali. Salva questo notebook: il `GridWorld` con ostacoli ti servirà come riferimento!